# Day 28 — Full Course Review

This notebook consolidates the entire 28-day course into a single reference. Each section is a one-page summary of a week's topics — runnable code for every concept.

**Use this notebook to:**
- Quickly recall syntax before the exam
- Identify gaps in your knowledge
- Run all patterns in one session

## Week 1 Review: Foundations

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

df = spark.createDataFrame([
    (1, "Alice",   "Engineering", "RO", 95000, "2022-03-15"),
    (2, "Bob",     "Marketing",   "RO", 72000, "2021-11-01"),
    (3, "Carol",   "Engineering", "UK", 110000,"2020-07-22"),
    (4, "Dave",    "Sales",       "DE", 58000, "2023-01-10"),
    (5, "Eve",     "Marketing",   "RO", 81000, "2021-05-30"),
    (6, None,      "Engineering", "UK", None,  None),
], ["id", "name", "dept", "country", "salary", "hire_date_str"])

# Schema
df.printSchema()

# Column operations
df.select(
    col("name"),
    (col("salary") * 1.1).alias("new_salary"),
    when(col("country") == "RO", "Romania").otherwise(col("country")).alias("country_full"),
    coalesce(col("name"), lit("Unknown")).alias("safe_name"),
).show()

# Filtering and nulls
df.filter(col("salary").isNotNull() & (col("salary") > 70000)).show()
df.dropna(subset=["name", "salary"]).show()
df.fillna({"name": "Unknown", "salary": 0}).show()

In [ ]:
# Spark SQL
df.createOrReplaceTempView("employees")
spark.sql("""
    SELECT dept,
           COUNT(*) AS n,
           AVG(salary) AS avg_salary
    FROM   employees
    WHERE  salary IS NOT NULL
    GROUP  BY dept
    HAVING COUNT(*) > 1
    ORDER  BY avg_salary DESC
""").show()

# Date functions
df2 = df.withColumn("hire_date", to_date(col("hire_date_str"), "yyyy-MM-dd"))
df2.select(
    col("hire_date"),
    year("hire_date").alias("yr"),
    datediff(current_date(), col("hire_date")).alias("days"),
).show()

## Week 2 Review: Advanced DataFrame API

In [ ]:
# Aggregations
df.groupBy("dept").agg(
    count("*").alias("n"),
    round(avg("salary"), 0).alias("avg_salary"),
    min("salary").alias("min"),
    max("salary").alias("max"),
).filter(col("n") > 1).show()

# Window functions
w = Window.partitionBy("dept").orderBy(col("salary").desc())
df.withColumn("rank",       rank().over(w)) \
  .withColumn("dense_rank", dense_rank().over(w)) \
  .withColumn("cumsum",     sum("salary").over(
      Window.partitionBy("dept").orderBy("salary")
            .rowsBetween(Window.unboundedPreceding, Window.currentRow)
  )) \
  .select("dept", "name", "salary", "rank", "dense_rank", "cumsum") \
  .orderBy("dept", "rank") \
  .show()

In [ ]:
# Joins
depts = spark.createDataFrame([
    ("Engineering", "Tech"), ("Marketing", "Ops"), ("Sales", "Ops")
], ["dept", "division"])

df.join(broadcast(depts), on="dept", how="inner").show()
df.join(depts, on="dept", how="left_semi").show()   # left cols only, has match
df.join(depts, on="dept", how="left_anti").show()   # no match only

# Collections
arr_df = spark.createDataFrame([
    (1, ["python", "sql", "spark"]),
    (2, ["java", "sql"]),
    (3, []),
], ["id", "skills"])

arr_df.select(
    col("id"),
    size(col("skills")).alias("n_skills"),
    array_contains(col("skills"), "sql").alias("knows_sql"),
    explode_outer(col("skills")).alias("skill"),
).show()

In [ ]:
# UDFs — row and pandas
from pyspark.sql.functions import udf, pandas_udf
import pandas as pd

@udf(returnType=StringType())
def grade(score):
    if score is None: return None
    return "A" if score >= 90000 else "B" if score >= 70000 else "C"

df.withColumn("grade", grade(col("salary"))).show()

@pandas_udf(DoubleType())
def normalize(s: pd.Series) -> pd.Series:
    return (s - s.min()) / (s.max() - s.min())

df.filter(col("salary").isNotNull()).withColumn("sal_norm", normalize(col("salary"))).show()

## Week 3 Review: Pipelines & Streaming

In [ ]:
from pyspark import StorageLevel
import tempfile, os

# Caching
cached = df.filter(col("salary").isNotNull()).cache()
cached.count()  # materialise
cached.groupBy("dept").agg(avg("salary")).show()
cached.unpersist()

# Storage levels
df.persist(StorageLevel.MEMORY_AND_DISK)
df.count()
df.unpersist()

# Accumulators
null_acc = spark.sparkContext.accumulator(0)
df.foreach(lambda r: null_acc.add(1 if r["salary"] is None else 0))
print(f"Null salaries: {null_acc.value}")

In [ ]:
# Structured Streaming — rate source → console
CKPT = tempfile.mkdtemp(prefix="review_ckpt_")
SRC  = tempfile.mkdtemp(prefix="review_src_")

# Create seed file
seed = spark.createDataFrame([
    ("ORD-1", "EU", 100, "2024-06-01 10:00:00"),
    ("ORD-2", "US", 200, "2024-06-01 10:01:00"),
    ("ORD-3", "EU", 150, "2024-06-01 10:06:00"),
], ["order_id", "region", "amount", "event_time"])

seed.write.mode("overwrite").json(SRC)

schema = StructType([
    StructField("order_id",   StringType(),  True),
    StructField("region",     StringType(),  True),
    StructField("amount",     IntegerType(), True),
    StructField("event_time", TimestampType(), True),
])

q = spark.readStream.schema(schema).json(SRC) \
    .withWatermark("event_time", "5 minutes") \
    .groupBy(window(col("event_time"), "5 minutes"), col("region")) \
    .agg(sum("amount").alias("total")) \
    .writeStream \
    .format("console") \
    .outputMode("update") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", CKPT) \
    .option("truncate", False) \
    .start()

q.awaitTermination(60)
print("Streaming review done.")

## Week 4 Review: Optimisation & Delta Lake

In [ ]:
# AQE and explain
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

df.join(broadcast(depts), on="dept").explain("simple")

# Partition pruning
PART_PATH = tempfile.mkdtemp()
df.write.mode("overwrite").partitionBy("country").parquet(PART_PATH)
pruned = spark.read.parquet(PART_PATH).filter(col("country") == "RO")
pruned.explain("formatted")
print(f"Pruned count: {pruned.count()}")

In [ ]:
# Delta Lake
from delta.tables import DeltaTable

DELTA_PATH = tempfile.mkdtemp(prefix="review_delta_")

# Create
df.filter(col("salary").isNotNull()) \
  .write.format("delta").mode("overwrite").save(DELTA_PATH)

dt = DeltaTable.forPath(spark, DELTA_PATH)

# Update
dt.update(col("dept") == "Sales", {"salary": lit(65000)})

# Merge (upsert)
updates = spark.createDataFrame([
    (1, "Alice", "Engineering", "RO", 102000, None),
    (7, "Grace", "HR",          "RO",  75000, None),
], ["id", "name", "dept", "country", "salary", "hire_date_str"])

dt.alias("t").merge(updates.alias("s"), "t.id = s.id") \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

# Time travel
print("Version 0:")
spark.read.format("delta").option("versionAsOf", 0).load(DELTA_PATH).show()

# History
dt.history().select("version", "operation").show()

## Exam Domain Coverage Summary

| Domain | Weight | Days covered |
|---|---|---|
| DataFrame API | 30% | 1–7, 8–14 |
| Architecture | 20% | 2, 15, 16, 17 |
| Spark SQL | 20% | 6, throughout |
| Optimisation | 10% | 14, 16, 17, 22, 23 |
| Streaming | 10% | 19, 20, 21 |
| Spark Connect | 5% | 26 |
| Pandas API | 5% | 18 |
| Delta Lake (bonus) | — | 24, 25 |

**Next:** Day 29 — Practice Exam Questions  
**Then:** Day 30 — Full Mock Exam